# Практическое занятие №3: Свёртка и корреляционный анализ сигналов

## 1. Функции Python для свёртки и корреляции

### 1.1. `np.convolve(x, h, mode)`

- `mode='full'` – полная линейная свёртка (длина $N_x+N_h-1$).
- `mode='same'` – центральная часть той же длины, что и $x$.
- `mode='valid'` – только те точки, где ядро полностью перекрывается с сигналом.

### 1.2. `np.correlate(x, y, mode)`

Аналогичные режимы. При `mode='full'` результат имеет длину $N_x+N_y-1$. Индекс 0 соответствует полному перекрытию сигналов.

Для получения сдвига, при котором достигается максимум корреляции, используется формула:
```python
corr = np.correlate(x, y, mode='full')
delay = np.argmax(corr) - (len(x) - 1)
```

---

## 2. Гауссовы окна и их свёртка

Гауссово окно можно сгенерировать с помощью `scipy.signal.windows.gaussian(M, std)`.

**Аналитическое свойство:** свёртка двух гауссовых функций с дисперсиями $\sigma_1^2$ и $\sigma_2^2$ даёт гауссову функцию с дисперсией $\sigma_1^2+\sigma_2^2$:
$$
(f_{\sigma_1} * f_{\sigma_2})(t) = f_{\sqrt{\sigma_1^2+\sigma_2^2}}(t).
$$
Это свойство используется в задании 1 для сравнения численного результата с теоретическим.

---

## 3. Фильтрация с помощью свёртки

**Прямоугольное окно (усреднение):**
```python
h_rect = signal.windows.boxcar(L)/L
```
Простое скользящее среднее.

**Гауссовское окно:**
```python
h_gauss = signal.windows.gaussian(L, std)
h_gauss = h_gauss / np.sum(h_gauss)  # нормализация
```

---

## 4. Поиск временной задержки с помощью кросс-корреляции

**Идея:** если $y[n] = x[n-d] + \text{шум}$, то кросс-корреляция $R_{xy}[m]$ имеет максимум при $m = d$.

**Алгоритм:**
1. Вычислить `corr = np.correlate(x, y, mode='full')`.
2. Найти индекс максимума: `idx = np.argmax(corr)`.
3. Задержка = `idx - (len(x)-1)`.

---

## 5. Обнаружение шаблона в зашумлённом сигнале

**Задача:** найти в длинном сигнале позицию, где содержится известный короткий шаблон.

**Метод:**
- Вычислить кросс-корреляцию между длинным сигналом и шаблоном (удобно использовать `mode='valid'`, чтобы избежать краевых эффектов).
- Пик корреляции указывает на позицию совпадения.

---

## 6. Работа с аудио в Python

**Чтение WAV-файла:**
```python
from scipy.io import wavfile
rate, data = wavfile.read('file.wav')
```
- `rate` – частота дискретизации (Гц).
- `data` – массив целых чисел. Для стерео форма `(N, 2)`.

**Нормализация в диапазон [-1, 1] (для 16-битного аудио):**
```python
data = data / 32767.0
```

**Прослушивание в Jupyter/Colab:**
```python
from IPython.display import Audio
Audio(data, rate=rate)
```

**Поиск фрагмента:**
- Вычислить кросс-корреляцию между полным сигналом и фрагментом.
- Найти индекс максимума.
- Вырезать соответствующий участок и убедиться в совпадении (визуально и на слух).

---

## 7. Рекомендации по выполнению заданий

- Для воспроизводимости результатов используйте `np.random.seed(42)` при генерации случайных чисел.
- Все графики должны быть подписаны (оси, заголовки, легенды).
- При анализе влияния шума на точность выполняйте несколько прогонов с разным `np.random.seed()` и усредняйте результаты.
- Для визуализации корреляции полезно показывать и сам сигнал, и корреляционную функцию.

---

## 8. Типичные ошибки и их избегание

1. **Путаница режимов `np.convolve`:** всегда проверяйте длину результата.
2. **Краевые эффекты:** при `mode='full'` корреляция даёт значения для всех возможных сдвигов, включая те, где перекрытие сигналов мало. Для поиска задержки это нормально, но при интерпретации нужно учитывать.
3. **Неправильное вычисление сдвига:** используйте `delay = argmax(corr) - (len(x)-1)`.
4. **Ненормализованное аудио:** перед прослушиванием убедитесь, что значения находятся в диапазоне [-1, 1].
5. **Сравнение с аналитикой:** при свёртке гауссовых функций не забывайте нормировать окна (сумма коэффициентов = 1), чтобы амплитуды соответствовали теоретическим.